# Tubes PCD: Klasifikasi Makanan Indonesia (Kelompok NMAX)
Komparasi **Model A** (pengolahan citra klasik) dan **Model B** (MobileNetV2 fine-tuning).

Isi notebook: (1) data, (2) praproses + segmentasi, (3) fitur HSV + GLCM, (4) baseline, (5) Model A Skenario 1 (prototipe + jarak) dan Skenario 2 (SVM), (6) ablasi, (7) evaluasi segmentasi (IoU/Dice), (8) Model B, (9) tabel perbandingan.

**Struktur folder yang diharapkan** (dataset Kaggle `rizkyyk/dataset-food-classification`, atur `DATA_DIR`):
```
DATA_DIR/train/<kelas>/*.jpg
DATA_DIR/val/<kelas>/*.jpg
DATA_DIR/test/<kelas>/*.jpg
```
Jika nama subfolder split di dataset berbeda, ubah `SPLIT_DIRS`.

In [6]:
# !pip install -q opencv-python scikit-image scikit-learn joblib tensorflow seaborn pandas matplotlib
import os, time, glob, warnings
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt, seaborn as sns
from pathlib import Path
from joblib import Parallel, delayed
from skimage.feature import graycomatrix, graycoprops
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from scipy.spatial.distance import cdist
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

# ====== KONFIGURASI ======
DATA_DIR   = Path('/dataset')          # ubah sesuai lokasi dataset
SPLIT_DIRS = {'train': 'train', 'val': 'val', 'test': 'test'}
EXCLUDE    = {'pizza', 'burger', 'french fries', 'french_fries', 'frenchfries'}  # 3 kelas dibuang
IMG_SIZE   = 224
N_JOBS     = -1
MASK_DIR   = None   # opsional: folder mask anotasi manual (nama file sama dgn citra, .png). None = lewati IoU/Dice

## 1. Muat data

In [7]:
EXTS = ('.jpg', '.jpeg', '.png')

def list_split(split):
    root = DATA_DIR / SPLIT_DIRS[split]
    paths, labels = [], []
    for cdir in sorted(p for p in root.iterdir() if p.is_dir()):
        if cdir.name.lower() in EXCLUDE:
            continue
        for f in sorted(cdir.rglob('*')):
            if f.suffix.lower() in EXTS:
                paths.append(str(f)); labels.append(cdir.name)
    return paths, labels

splits = {s: list_split(s) for s in SPLIT_DIRS}
CLASSES = sorted(set(splits['train'][1]))
cls2id = {c: i for i, c in enumerate(CLASSES)}
data = {s: (p, np.array([cls2id[l] for l in y])) for s, (p, y) in splits.items()}

print('Kelas:', CLASSES, f'({len(CLASSES)} kelas)')
for s in data: print(s, len(data[s][0]), 'citra')

FileNotFoundError: [WinError 3] The system cannot find the path specified: '\\dataset\\train'

## 2. Praproses, segmentasi, dan ekstraksi fitur

In [ ]:
KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
H_BINS = (8, 8, 8)       # histogram HSV 3D -> 512 bin
GLCM_LEVELS = 32
GLCM_PROPS = ['contrast', 'homogeneity', 'energy', 'correlation']   # 4 nilai GLCM

def read_img(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(path)
    return img

def preprocess(img, use_blur=True):
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)   # resize selalu dilakukan
    if use_blur:
        img = cv2.GaussianBlur(img, (5, 5), 0)
    return img

def segment(img_bgr, use_seg=True):
    """Otsu pada kanal saturasi HSV + morfologi (opening lalu closing). Return mask uint8 {0,255}."""
    if not use_seg:
        return np.full(img_bgr.shape[:2], 255, np.uint8)
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    _, m = cv2.threshold(hsv[..., 1], 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    m = cv2.morphologyEx(m, cv2.MORPH_OPEN, KERNEL)
    m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, KERNEL)
    if m.mean() / 255 < 0.05:          # segmentasi gagal -> pakai seluruh citra
        m = np.full_like(m, 255)
    return m

def color_hist(img_bgr, mask):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    h = cv2.calcHist([hsv], [0, 1, 2], mask, H_BINS, [0, 180, 0, 256, 0, 256]).flatten()
    return h / (h.sum() + 1e-12)

def glcm_feats(img_bgr, mask):
    g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    ys, xs = np.where(mask > 0)
    y0, y1, x0, x1 = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    g, m = g[y0:y1, x0:x1], mask[y0:y1, x0:x1]
    q = (g.astype(np.float32) / 256 * (GLCM_LEVELS - 1)).astype(np.uint8) + 1   # level 1..L-1; 0 = latar
    q[m == 0] = 0
    P = graycomatrix(q, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
                     levels=GLCM_LEVELS + 1, symmetric=True, normed=False)
    P = P[1:, 1:, :, :].astype(np.float64)            # buang level 0 (piksel di luar area makanan)
    s = P.sum(axis=(0, 1), keepdims=True); s[s == 0] = 1
    P = P / s
    return np.array([graycoprops(P, p).mean() for p in GLCM_PROPS])

def extract(path, use_blur=True, use_seg=True, return_mask=False):
    img = preprocess(read_img(path), use_blur)
    mask = segment(img, use_seg)
    out = (color_hist(img, mask), glcm_feats(img, mask))
    return (out + (mask,)) if return_mask else out

def extract_all(paths, **kw):
    res = Parallel(n_jobs=N_JOBS)(delayed(extract)(p, **kw) for p in paths)
    return np.stack([r[0] for r in res]), np.stack([r[1] for r in res])

# Contoh visual segmentasi
fig, ax = plt.subplots(2, 6, figsize=(16, 6))
rng = np.random.RandomState(0)
for i, idx in enumerate(rng.choice(len(data['train'][0]), 6, replace=False)):
    p = data['train'][0][idx]
    img = preprocess(read_img(p))
    m = segment(img)
    ax[0, i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB)); ax[0, i].set_title(CLASSES[data['train'][1][idx]], fontsize=9)
    ax[1, i].imshow(cv2.bitwise_and(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), cv2.cvtColor(img, cv2.COLOR_BGR2RGB), mask=m))
    ax[0, i].axis('off'); ax[1, i].axis('off')
plt.tight_layout(); plt.show()

## 3. Fungsi klasifikasi (baseline, Skenario 1, Skenario 2) dan metrik

In [ ]:
def metrics(y_true, y_pred):
    return {'acc': accuracy_score(y_true, y_pred), 'f1_macro': f1_score(y_true, y_pred, average='macro')}

# ---------- Skenario 1: prototipe per kelas + jarak gabungan ----------
class PrototypeClassifier:
    """d = w * Bhattacharyya(hist HSV) + (1 - w) * Euclidean(GLCM z-score).
    Prototipe = rata-rata fitur semua citra latih pada kelas tsb. w dipilih dari data validasi."""
    def __init__(self, w_grid=(0.3, 0.5, 0.7)):
        self.w_grid = w_grid; self.w = None
    def fit(self, H, G, y):
        self.mu, self.sd = G.mean(0), G.std(0) + 1e-9
        Z = (G - self.mu) / self.sd
        self.classes = np.unique(y)
        self.PH = np.stack([H[y == c].mean(0) for c in self.classes])
        self.PG = np.stack([Z[y == c].mean(0) for c in self.classes])
        return self
    def _dists(self, H, G):
        bc = np.sqrt(H) @ np.sqrt(self.PH).T
        db = np.sqrt(np.clip(1 - bc, 0, None))                      # Bhattacharyya distance
        de = cdist((G - self.mu) / self.sd, self.PG)                # Euclidean pada GLCM z-score
        return db, de
    def tune_w(self, H, G, y):
        db, de = self._dists(H, G)
        scores = {w: accuracy_score(y, self.classes[(w * db + (1 - w) * de).argmin(1)]) for w in self.w_grid}
        self.w = max(scores, key=scores.get)
        return scores
    def predict(self, H, G, w=None):
        w = self.w if w is None else w
        db, de = self._dists(H, G)
        return self.classes[(w * db + (1 - w) * de).argmin(1)]

# ---------- Skenario 2: SVM ----------
def feat_svm(H, G):
    return np.hstack([np.sqrt(H), G])        # akar histogram (Hellinger) + 4 nilai GLCM

def fit_svm(Htr, Gtr, ytr, Hva, Gva, yva):
    sc = StandardScaler().fit(feat_svm(Htr, Gtr))
    Xtr, Xva = sc.transform(feat_svm(Htr, Gtr)), sc.transform(feat_svm(Hva, Gva))
    best, best_acc = None, -1
    for C in [1, 10, 100]:
        for g in ['scale', 1e-3, 1e-2]:
            m = SVC(C=C, gamma=g, kernel='rbf', random_state=SEED).fit(Xtr, ytr)
            a = accuracy_score(yva, m.predict(Xva))
            if a > best_acc: best, best_acc, params = m, a, (C, g)
    return sc, best, params

# ---------- Baseline: histogram warna citra utuh + k-NN ----------
def run_baseline(feats):
    Htr, _ = feats['train']; Hva, _ = feats['val']; Hte, _ = feats['test']
    ytr, yva, yte = data['train'][1], data['val'][1], data['test'][1]
    best_k, best_a = 1, -1
    for k in [1, 3, 5, 7, 9]:
        a = accuracy_score(yva, KNeighborsClassifier(k, metric='euclidean').fit(np.sqrt(Htr), ytr).predict(np.sqrt(Hva)))
        if a > best_a: best_k, best_a = k, a
    knn = KNeighborsClassifier(best_k).fit(np.sqrt(Htr), ytr)
    return metrics(yte, knn.predict(np.sqrt(Hte))), best_k

# ---------- Satu konfigurasi Model A (kedua skenario) ----------
def run_model_a(feats, use_color=True, use_texture=True):
    Htr, Gtr = feats['train']; Hva, Gva = feats['val']; Hte, Gte = feats['test']
    ytr, yva, yte = data['train'][1], data['val'][1], data['test'][1]
    out = {}
    # Skenario 1
    pc = PrototypeClassifier().fit(Htr, Gtr, ytr)
    if use_color and use_texture:
        pc.tune_w(Hva, Gva, yva); w = pc.w
    else:
        w = 1.0 if use_color else 0.0           # hanya warna / hanya tekstur
        pc.w = w
    out['S1'] = metrics(yte, pc.predict(Hte, Gte, w)); out['S1']['w'] = w
    # Skenario 2 (fitur yang dimatikan dinolkan agar tidak berkontribusi)
    def mask_feats(H, G):
        return (H if use_color else np.zeros_like(H)), (G if use_texture else np.zeros_like(G))
    a, b, c = mask_feats(Htr, Gtr), mask_feats(Hva, Gva), mask_feats(Hte, Gte)
    sc, svm, params = fit_svm(*a, ytr, *b, yva)
    out['S2'] = metrics(yte, svm.predict(sc.transform(feat_svm(*c)))); out['S2']['params'] = params
    return out, (pc, sc, svm)

## 4. Ekstraksi fitur untuk semua konfigurasi
Fitur dihitung sekali per konfigurasi praproses/segmentasi lalu dipakai ulang.

In [ ]:
CONFIGS = {
    'usulan (blur + seg)':   dict(use_blur=True,  use_seg=True),
    'tanpa praproses':       dict(use_blur=False, use_seg=True),
    'tanpa segmentasi':      dict(use_blur=True,  use_seg=False),
    'baseline (citra utuh)': dict(use_blur=False, use_seg=False),
}
FEATS = {}
for name, kw in CONFIGS.items():
    t0 = time.time()
    FEATS[name] = {s: extract_all(data[s][0], **kw) for s in data}
    print(f'{name:25s} selesai {time.time()-t0:.0f} dtk')

## 5. Baseline, Model A (metode usulan), dan skenario ablasi

In [ ]:
rows = []

# Baseline
bl, k = run_baseline(FEATS['baseline (citra utuh)'])
rows.append({'Skenario': f'Baseline (hist HSV utuh + k-NN, k={k})', **bl})

# Metode usulan
res_full, (pc_full, sc_full, svm_full) = run_model_a(FEATS['usulan (blur + seg)'])
rows += [
    {'Skenario': f"Model A - S1 prototipe (w={res_full['S1']['w']})", **{k: res_full['S1'][k] for k in ('acc','f1_macro')}},
    {'Skenario': 'Model A - S2 SVM',                                    **{k: res_full['S2'][k] for k in ('acc','f1_macro')}},
]

# Ablasi (dilakukan pada kedua skenario)
ablasi = [
    ('Ablasi 1: tanpa praproses',      'tanpa praproses',   True,  True),
    ('Ablasi 2: tanpa segmentasi',     'tanpa segmentasi',  True,  True),
    ('Ablasi 3: tanpa fitur warna',    'usulan (blur + seg)', False, True),
    ('Ablasi 4: tanpa fitur tekstur',  'usulan (blur + seg)', True,  False),
]
for label, cfg, uc, ut in ablasi:
    r, _ = run_model_a(FEATS[cfg], use_color=uc, use_texture=ut)
    rows.append({'Skenario': f'{label} - S1', 'acc': r['S1']['acc'], 'f1_macro': r['S1']['f1_macro']})
    rows.append({'Skenario': f'{label} - S2', 'acc': r['S2']['acc'], 'f1_macro': r['S2']['f1_macro']})

df_a = pd.DataFrame(rows)
df_a['acc'] = (df_a['acc'] * 100).round(2); df_a['f1_macro'] = df_a['f1_macro'].round(4)
df_a

In [ ]:
# Penurunan akurasi tiap ablasi terhadap metode usulan -> tahap paling berpengaruh
ref = {'S1': res_full['S1']['acc'] * 100, 'S2': res_full['S2']['acc'] * 100}
d = df_a[df_a['Skenario'].str.startswith('Ablasi')].copy()
d['skenario'] = d['Skenario'].str[-2:]
d['tahap'] = d['Skenario'].str.replace(r' - S[12]$', '', regex=True)
d['selisih_poin'] = d.apply(lambda r: round(r['acc'] - ref[r['skenario']], 2), axis=1)
print(d.pivot(index='tahap', columns='skenario', values='selisih_poin'))
print('\nSelisih negatif terbesar = tahap dengan kontribusi terbesar.')

## 6. Evaluasi segmentasi (IoU dan Dice) pada subset citra uji beranotasi
Isi `MASK_DIR` dengan folder mask manual (putih = makanan) yang namanya sama dengan citra (`.png`). Lewati jika belum ada.

In [ ]:
def seg_scores(n_max=100):
    if MASK_DIR is None:
        print('MASK_DIR belum diisi -> evaluasi segmentasi dilewati.'); return None
    ious, dices = [], []
    for p in data['test'][0]:
        mp = Path(MASK_DIR) / (Path(p).stem + '.png')
        if not mp.exists(): continue
        gt = cv2.resize(cv2.imread(str(mp), 0), (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST) > 127
        pr = segment(preprocess(read_img(p))) > 0
        inter = (gt & pr).sum(); union = (gt | pr).sum()
        ious.append(inter / (union + 1e-9)); dices.append(2 * inter / (gt.sum() + pr.sum() + 1e-9))
        if len(ious) >= n_max: break
    if not ious: print('Tidak ada mask yang cocok.'); return None
    print(f'n={len(ious)}  IoU={np.mean(ious):.3f}  Dice={np.mean(dices):.3f}')
    return np.mean(ious), np.mean(dices)
seg_res = seg_scores()

## 7. Model B: MobileNetV2 (pretrained, fine-tuning)

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
tf.keras.utils.set_random_seed(SEED)

BATCH = 32
def make_ds(split, shuffle):
    ds = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR / SPLIT_DIRS[split], labels='inferred', label_mode='int',
        class_names=CLASSES, image_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH,
        shuffle=shuffle, seed=SEED)
    return ds.prefetch(tf.data.AUTOTUNE)
ds_tr, ds_va, ds_te = make_ds('train', True), make_ds('val', False), make_ds('test', False)

aug = tf.keras.Sequential([layers.RandomFlip('horizontal'), layers.RandomRotation(0.08),
                           layers.RandomZoom(0.1), layers.RandomContrast(0.1)])
base = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights='imagenet')
base.trainable = False
inp = layers.Input((IMG_SIZE, IMG_SIZE, 3))
x = aug(inp); x = preprocess_input(x)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x); x = layers.Dropout(0.3)(x)
out = layers.Dense(len(CLASSES), activation='softmax')(x)
model_b = models.Model(inp, out)

model_b.compile(optimizers.Adam(1e-3), 'sparse_categorical_crossentropy', ['accuracy'])
cb = [callbacks.EarlyStopping(patience=3, restore_best_weights=True, monitor='val_accuracy'),
      callbacks.ReduceLROnPlateau(patience=2, factor=0.3)]
h1 = model_b.fit(ds_tr, validation_data=ds_va, epochs=8, callbacks=cb)

# Fine-tuning: buka 30 layer teratas
base.trainable = True
for l in base.layers[:-30]: l.trainable = False
model_b.compile(optimizers.Adam(1e-5), 'sparse_categorical_crossentropy', ['accuracy'])
h2 = model_b.fit(ds_tr, validation_data=ds_va, epochs=12, callbacks=cb)

In [ ]:
y_true_b = np.concatenate([y.numpy() for _, y in ds_te])
prob = model_b.predict(ds_te, verbose=0)
y_pred_b = prob.argmax(1)
res_b = metrics(y_true_b, y_pred_b)
print(res_b)
print(classification_report(y_true_b, y_pred_b, target_names=CLASSES, digits=3))

## 8. Perbandingan akhir, confusion matrix, dan waktu proses per citra

In [ ]:
yte = data['test'][1]
final = pd.concat([df_a, pd.DataFrame([{'Skenario': 'Model B - MobileNetV2 fine-tuning',
                                        'acc': round(res_b['acc']*100, 2), 'f1_macro': round(res_b['f1_macro'], 4)}])],
                  ignore_index=True)
final['acc'] = final['acc'].astype(float)
bl_acc = final.loc[0, 'acc']
final['selisih_vs_baseline (poin)'] = (final['acc'] - bl_acc).round(2)
final['memenuhi target (acc>=80, F1>=0.80)'] = (final['acc'] >= 80) & (final['f1_macro'] >= 0.80)
display(final)
final.to_csv('hasil_perbandingan.csv', index=False)

# Confusion matrix Model A (S1, S2) dan Model B
Htr, Gtr = FEATS['usulan (blur + seg)']['test']
pred_s1 = pc_full.predict(Htr, Gtr)
pred_s2 = svm_full.predict(sc_full.transform(feat_svm(Htr, Gtr)))
fig, ax = plt.subplots(1, 3, figsize=(21, 6))
for a, (t, pr) in zip(ax, [('Model A - S1', pred_s1), ('Model A - S2 (SVM)', pred_s2), ('Model B - MobileNetV2', y_pred_b)]):
    sns.heatmap(confusion_matrix(yte, pr), annot=True, fmt='d', cmap='Blues', cbar=False,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=a)
    a.set_title(t); a.tick_params(axis='x', rotation=90)
plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=150); plt.show()

In [ ]:
# Waktu proses per citra (target <= 3 detik)
sample = data['test'][0][:30]
def time_a():
    t0 = time.time()
    for p in sample:
        H, G = extract(p)
        pc_full.predict(H[None], G[None])
    return (time.time() - t0) / len(sample)
def time_b():
    t0 = time.time()
    for p in sample:
        im = cv2.cvtColor(cv2.resize(read_img(p), (IMG_SIZE, IMG_SIZE)), cv2.COLOR_BGR2RGB)
        model_b.predict(im[None].astype('float32'), verbose=0)
    return (time.time() - t0) / len(sample)
print(f'Model A (S1): {time_a():.3f} dtk/citra')
print(f'Model B     : {time_b():.3f} dtk/citra')

In [ ]:
# Simpan model untuk aplikasi demo
import joblib
model_b.save('model_b_mobilenetv2.keras')
joblib.dump({'proto': pc_full, 'scaler': sc_full, 'svm': svm_full, 'classes': CLASSES}, 'model_a.joblib')